In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# 1. Load Data
train_data_raw = pd.read_csv("../MLP/MNIST_CSV/mnist_train.csv")
test_data_raw = pd.read_csv("../MLP/MNIST_CSV/mnist_test.csv")

# 2. Shuffle and Split Validation Set
val_size = 10000
train_data_raw = train_data_raw.sample(frac=1,random_state=42).reset_index(drop=True)

val_data_raw = train_data_raw.iloc[:val_size]
train_data_raw = train_data_raw.iloc[val_size:]

# 3. Separate Features (X) and Labels (y), then Normalize (Zero centered, Fixed Range (-0.5 to 0.5))

def normalize_and_split(data):
    X = data.iloc[:,1:]
    X = (X / 255) - 0.5 
    y = data.iloc[:,0]
    return X, y
x_train, y_train = normalize_and_split(train_data_raw)
x_val, y_val = normalize_and_split(val_data_raw)
x_test, y_test = normalize_and_split(test_data_raw)

# 4. One-Hot Encode Training Labels
y_train_ohe = np.zeros((y_train.shape[0], 10))
y_train_ohe[np.arange(y_train.shape[0]), y_train] = 1

y_test_ohe = np.zeros((y_test.shape[0], 10))
y_test_ohe[np.arange(y_test.shape[0]), y_test] = 1

Y_val_ohe = np.zeros((y_val.shape[0], 10))
Y_val_ohe[np.arange(y_val.shape[0]), y_val] = 1


print(f"Training data shape: {x_train.shape}")
print(f"Validation data shape: {x_val.shape}")
print(f"Test data shape: {x_test.shape}")

Training data shape: (49999, 784)
Validation data shape: (10000, 784)
Test data shape: (9999, 784)


In [ ]:
class MLP_PyTorch(nn.Module):

    def __init__(self, input_size, fc1_size, fc2_size, output_size, initialization='He', hidden_activation_function='ReLU', output_activation_function='Softmax'):
        super().__init__()

        # Define the layers of the MLP
        self.fc1 = nn.Linear(input_size, fc1_size)
        self.fc2 = nn.Linear(fc1_size, fc2_size)
        self.output_layer = nn.Linear(fc2_size, output_size)

        # Store initialization and activation function choices
        self.initialization = initialization
        self.hidden_activation_function = hidden_activation_function
        self.output_activation_function = output_activation_function

        # Define activation functions
        hidden_activation_dict = {
            'ReLU': nn.ReLU(),
            'LeakyReLU': nn.LeakyReLU(negative_slope=0.01),
            'GELU': nn.GELU(),
            'SiLU': nn.SiLU()
        }
        output_activation_dict = {
            'Tanh': nn.Tanh(),
            'Sigmoid': nn.Sigmoid(),
            'Softmax': nn.Softmax(dim=1)
        }

        # Set the activation functions based on input
        if hidden_activation_function in hidden_activation_dict:
            self.hidden_activation = hidden_activation_dict[hidden_activation_function]
        else:
            raise ValueError(f"Invalid hidden activation function: {hidden_activation_function}")
        
        if output_activation_function in output_activation_dict:
            self.output_activation = output_activation_dict[output_activation_function] 
        else:
            raise ValueError(f"Invalid output activation function: {output_activation_function}")

        self.apply(self.init_weights)

    def forward(self, x):

        # Passing input to the first fully connected layer
        x = self.fc1(x) 

        # Passing the output of the first layer to the ReLU activation function
        x = self.hidden_activation(x)

        # Passing the output of the activation function to the second fully connected layer
        x = self.fc2(x)

        # Passing the output of the second layer to the ReLU activation function
        x = self.hidden_activation(x)

        # Passing the output of the activation function to the output layer
        x = self.output_layer(x)
        x = self.output_activation(x)

        return x 
    
    def init_weights(self, module):
        if isinstance(module, nn.Linear):
            if self.initialization == 'He':
                nn.init.kaiming_normal_(module.weight, nonlinearity = 'relu' if self.hidden_activation_function in ['GELU', 'SiLU'] else self.hidden_activation_function.lower().replace('leakyrelu', 'leaky_relu')
                )
    

In [ ]:
model = MLP_PyTorch(784,256,128,10,initialization = 'He', hidden_activation_function = 'ReLU', output_activation_function = 'Softmax')

model.back
